# Data Cleaning
Use prem data for exploratory data analysis

In [1]:
# packages
import pandas as pd

# Reusable code lives in src/footy_prediction (install with `pip install -e .` from the repo root).
from footy_prediction.data_loading import load_season, load_season_xg
from footy_prediction.feature_engineering import (
    build_rolling_xg_features,
    build_team_name_mapping,
    join_xg_features,
    upload_features,
)
from footy_prediction.paths import FEATURES_DIR, UNDERSTAT_DIR
# One DuckDB connection shared with the package functions.
from footy_prediction.sql_utils import con

In [5]:
xg_df = load_season_xg(
    league_codes=["E0", "SP1", "D1", "I1", "F1"],
    seasons=[19, 27],
)

Processing xG E0 season 1920...
Processing xG E0 season 2021...
Processing xG E0 season 2122...
Processing xG E0 season 2223...
Processing xG E0 season 2324...
Processing xG E0 season 2425...
Processing xG E0 season 2526...
Processing xG E0 season 2627...
Processing xG SP1 season 1920...
Processing xG SP1 season 2021...
Processing xG SP1 season 2122...
Processing xG SP1 season 2223...
Processing xG SP1 season 2324...
Processing xG SP1 season 2425...
Processing xG SP1 season 2526...
Processing xG SP1 season 2627...
Processing xG D1 season 1920...
Processing xG D1 season 2021...
Processing xG D1 season 2122...
Processing xG D1 season 2223...
Processing xG D1 season 2324...
Processing xG D1 season 2425...
Processing xG D1 season 2526...
Processing xG D1 season 2627...
Processing xG I1 season 1920...
Processing xG I1 season 2021...
Processing xG I1 season 2122...
Processing xG I1 season 2223...
Processing xG I1 season 2324...
Processing xG I1 season 2425...
Processing xG I1 season 2526...


In [8]:
# names_mapping.sql: builds team_name_mapping from xg_df (needed by feature_extraction.sql).
build_team_name_mapping()

# # create path to sql script which does all the feature engineering.
# sql_file = Path(
#     "/accounts/masters/gautierep/footy_prediction/footy-prediction"
# ) / "feature_extraction.sql"

# # reads the entire sql file into python as a text string and executes
# con.execute(sql_file.read_text())


# con.execute(sql_file.read_text())
# # Query the final SQL feature table prematch_sql made by the script
# prematch_sql = con.sql("""
#     SELECT *
#     FROM prematch_sql
#     ORDER BY match_id
#     LIMIT 10
# """).df()

# prematch_features = prematch_sql[[
#     # Info about match
#     "match_concat",
#     "match_id",
#     "kickoff",
#     "HomeTeam",
#     "AwayTeam",
#     "FTR",
#     "HS",
#     "AS",
#     # Recent form
#     "home_points_last_5",
#     "away_points_last_5",
#     "home_points_last_10",
#     "away_points_last_10",
#     # venue strength
#     "home_points_home_last_5",
#     "away_points_away_last_5",
#     "home_goals_for_home_last_5",
#     "away_goals_for_away_last_5",
#     "home_points_home_last_10",
#     "away_points_away_last_10",
#     "home_goals_for_home_last_10",
#     "away_goals_for_away_last_10",
#     "market_home_prob_fair",
#     "market_draw_prob_fair",
#     "market_away_prob_fair",
#     # difference in recent form
#     "goals_difference_last_5",
#     "points_difference_last_5",
#     # extra features
#     "home_shots_on_target_for_last_5",
#     "away_shots_on_target_for_last_5",
#     "home_fouls_for_last_5",
#     "away_fouls_for_last_5",
#     "home_corners_for_last_5",
#     "away_corners_for_last_5",
# ]]

# prematch_features


In [9]:
features_df = upload_features(
    league_codes=["E0", "SP1", "D1", "I1", "F1"],
    seasons=[19, 27],
)

Processing E0 season 1920...
Processing E0 season 2021...
Processing E0 season 2122...
Processing E0 season 2223...
Processing E0 season 2324...
Processing E0 season 2425...
Processing E0 season 2526...
Processing E0 season 2627...
Processing SP1 season 1920...
Processing SP1 season 2021...
Processing SP1 season 2122...
Processing SP1 season 2223...
Processing SP1 season 2324...
Processing SP1 season 2425...
Processing SP1 season 2526...
Processing SP1 season 2627...
Processing D1 season 1920...
Processing D1 season 2021...
Processing D1 season 2122...
Processing D1 season 2223...
Processing D1 season 2324...
Processing D1 season 2425...
Processing D1 season 2526...
Processing D1 season 2627...
Processing I1 season 1920...
Processing I1 season 2021...
Processing I1 season 2122...
Processing I1 season 2223...
Processing I1 season 2324...
Processing I1 season 2425...
Processing I1 season 2526...
Processing I1 season 2627...
Processing F1 season 1920...
Processing F1 season 2021...
Proces

In [22]:
features_df.shape


(12653, 236)

In [24]:
# TEST MAPPING TABLE
# Execute the SQL script
build_team_name_mapping()

# test that it is indeed many to one mapping - should return nothing
con.sql("""
    SELECT home_team, count(*) as count from (SELECT home_team, home_team_code, count(*) as count
    FROM xg_df
    GROUP BY home_team, home_team_code) group by home_team having count > 1 
""").df() 

con.sql("SELECT * FROM team_name_mapping LIMIT 10").df()


,team_name,team_code,count
0,Southampton,SOU,95
1,Wolverhampton Wanderers,WOL,133
2,Athletic Club,ATH,152
3,Celta Vigo,CEL,152
4,Villarreal,VIL,152
5,Alaves,ALA,133
6,Espanyol,ESP,114
7,SD Huesca,HUE,19
8,Real Oviedo,OVI,19
9,Fortuna Duesseldorf,FOR,17


In [ ]:
# Register the saved feature table on the same DuckDB connection as the xG data.
# PUT XG TABLE ON DUCKDB
# get xg table
xg_df = pd.read_csv(UNDERSTAT_DIR / "xg_E0_SP1_D1_I1_F1_1927.csv")
# put it on duckdb server
con.register("xg_df", xg_df)

# Execute the xG join SQL script.
features_xg_df = join_xg_features()

features_xg_df = pd.read_csv(FEATURES_DIR / "features_data_xg_rolling.csv")
features_xg_df[(features_xg_df['home_xg_last_5'].isnull()) & (features_xg_df['home_fouls_for_last_5'].notnull())].shape

(169, 21)

In [ ]:
# rolling_xg.sql: rolling xG over each team's previous 5 matches -> model-ready table.
# scripts/build_features.py saves this as data/features/features_data_xg_rolling.csv.
features_data_xg = build_rolling_xg_features()
features_data_xg.shape

# APPENDIX

## Feature-engineering logic

`sql/feature_extraction.sql` (run per league-season by `upload_features`) creates variables that could have been known immediately before each match. It starts by combining the date and time into a single `kickoff` column so every match can be ordered chronologically.

The average bookmaker odds are converted into implied probabilities using `1 / odds`. These probabilities include the bookmaker's margin, so the three values are divided by their total (`overround`) to create normalized market probabilities that sum to one. These are useful baseline features because they summarize the market's pre-match expectations.

The match table is then reshaped into team-history rows. Each match produces one row from the home team's perspective and one from the away team's perspective. This makes it possible to calculate each team's previous points, goals, shots, and shots on target consistently, regardless of whether the team played at home or away.

For each team, the function calculates:

- Rest days since the previous match
- Mean points over the previous 5 and 10 matches
- Mean goals scored over the previous 5 and 10 matches
- Mean goals conceded over the previous 5 and 10 matches
- Mean shots and shots on target over the previous 5 and 10 matches

The important leakage safeguard is the window frame `ROWS BETWEEN n PRECEDING AND 1 PRECEDING` (the SQL equivalent of `.shift(1)` before a rolling mean). It ends one match before the current one. Therefore, a match's own result is never used to create its predictors. The first match for each team has missing historical features because no earlier information exists.

Finally, the home and away history features are merged back onto the original match rows. Difference features, such as `points_difference_last_5`, compare the recent form of the two teams. The resulting `prematch_sql` table (stacked across seasons into `features_df`) contains both the original target column, `FTR`, and the pre-match predictors for later modelling.

1. Recent form
- overall_points_last_5
- overall_points_last_10
- overall_goals_scored_last_5
- overall_goals_conceded_last_5
- overall_shots_last_5
- overall_shots_on_target_last_5
- overall_corners_last_5

2. Venue strength
- home_points_last_5
- home_points_last_10
- home_goals_scored_last_5
- home_goals_conceded_last_5
- away_points_last_5
- away_points_last_10
- away_goals_scored_last_5
- away_goals_conceded_last_5

3. Matchup differences
- home_team_goals_scored_last_5
- away_team_goals_scored_last_5

# Data validation

Take Tottenham vs Man U on 2025-11-08 (E0, 2025/26)

Manually calc away_points_last_5, home_points_home_last_5

below we see that sums are equivalent

In [ ]:
list(features_df.columns)

validation_match = (
    (features_df["league"] == "E0")
    & (features_df["season"] == "2025/26")
    & (features_df["HomeTeam"] == "Tottenham")
    & (features_df["AwayTeam"] == "Man United")
)

features_df[validation_match][['match_id',
    'kickoff',
    'HomeTeam',
    'AwayTeam',
    'FTR',
    'home_points_last_5',
    'away_points_last_5',
    'home_points_home_last_5',
    'away_points_away_last_5',
    'points_difference_last_5',
    'goals_difference_last_5',
    'market_home_prob_fair',
    'market_draw_prob_fair',
    'market_away_prob_fair']].head(10)

In [ ]:
df_2526 = load_season("E0", 2526)

temp = df_2526[((df_2526["HomeTeam"] == "Tottenham") | (df_2526["AwayTeam"] == "Tottenham") | 
        (df_2526["HomeTeam"] == "Man United") | (df_2526["AwayTeam"] == "Man United"))].copy()

temp["kickoff"] = pd.to_datetime(
    temp["Date"].astype(str) + " " + temp["Time"].fillna("00:00").astype(str),
    dayfirst=True,
    errors="coerce",
)

temp = temp[(temp["kickoff"] > "2025-08-10") & (temp["kickoff"] < "2025-11-09")]
temp[temp["HomeTeam"] == "Tottenham"][["kickoff", "HomeTeam", "AwayTeam", "FTR"]]